# **Evaluate RAG Application** with RAGAS

In [ ]:
# Uninstall conflicting packages first to ensure a clean state
!pip uninstall -y ragas langchain langchain-core langchain-community langchain-chroma langchain-openai langchain-text-splitters langsmith langchain-google-genai

# Install a compatible set of versions
!pip install langchain==0.1.20 ragas langchain-chroma langchain-openai langchain-community langchain-google-genai datasets pypdf

Found existing installation: langchain 1.3.17
Uninstalling langchain-1.3.17:
  Successfully uninstalled langchain-1.3.17
Found existing installation: langchain-core 1.6.0
Uninstalling langchain-core-1.6.0:
  Successfully uninstalled langchain-core-1.6.0
Found existing installation: langsmith 0.11.1
Uninstalling langsmith-0.11.1:
  Successfully uninstalled langsmith-0.11.1
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 83.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
INFO: pip is looking at multiple versions of langchain-chroma to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of langchain-chroma to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of langchain

In [ ]:
import os
from google.colab import userdata
from pydantic import SecretStr # Import SecretStr for type checking

import warnings
warnings.filterwarnings('ignore')

# Safely extract the API key, handling both SecretStr and plain string types
api_key_value = userdata.get('GEMINI_API_KEY')

if isinstance(api_key_value, SecretStr):
    os.environ["GOOGLE_API_KEY"] = api_key_value.get_secret_value()
else:
    os.environ["GOOGLE_API_KEY"] = api_key_value

# Initialize Google Gemini LLM

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from google.colab import userdata
import os

# Initialize the GenAI model
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.7,
    # The API key will be automatically picked up from the GOOGLE_API_KEY environment variable.
)

# Initialize Embedding Model

In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from google.colab import userdata
import os

# Initialize the embedding model
embedding_model = GoogleGenerativeAIEmbeddings(
    model="models/embedding-001", # Changed to a stable Gemini embedding model
    # The API key will be automatically picked up from the GOOGLE_API_KEY environment variable.
)

# Load PDF Document

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

# Load the PDF document
loader = PyPDFLoader("/content/codeprolk.pdf")

docs = loader.load()

In [ ]:
len(docs)

4

# Split Documents into Chunks

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Initialize the text splitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=20)

# Split the documents into chunks
documents = text_splitter.split_documents(docs)

In [ ]:
len(documents)

28

# Create Vector Store and Retriever

In [ ]:
from langchain_chroma import Chroma

# Create a vector store from the document chunks
vectorstore = Chroma.from_documents(documents=documents, embedding=embedding_model)

In [ ]:
# Create a retriever from the vector store
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

NameError: name 'vectorstore' is not defined

# Define Prompt Template

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

# Define a message template for the chatbot
message = """
Answer this question using the provided context only.

{question}

Context:
{context}

"""

# Create a chat prompt template from the message
prompt = ChatPromptTemplate.from_messages([("human", message)])


# Create Retrieval-Augmented Generation (RAG) chain

In [ ]:
from langchain_core.output_parsers import StrOutputParser

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# Invoke RAG Chain with Example Questions

In [ ]:
response = chain.invoke("who is codeprolk")
print(response)

# Evaluate RAG Application

In [ ]:
import pandas as pd
test_data = pd.read_csv("/content/test_data.csv")
test_data

In [ ]:
questions = test_data ["question"].to_list()
ground_truth = test_data ["answer"].to_list()

In [ ]:
questions

In [ ]:
ground_truth

In [ ]:
data = {"question" : [], "answer": [], "contexts" : [], "ground_truth" : ground_truth}

In [ ]:
for query in questions:
  data["question"].append(query)
  data["answer"].append(chain.invoke(query))
  data["contexts"].append([doc.page_content for doc in retriever.invoke(query)])


In [ ]:
data

In [ ]:
# to evaluate from ragas
from datasets import Dataset

dataset = Dataset.from_dict(data)

In [ ]:
dataset

In [ ]:
dataset [0]

In [ ]:
from ragas.metrics import context_precision, context_recall, answer_relevancy, faithfulness
from ragas import evaluate
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

# Wrap the Gemini LLM and embedding model so ragas uses them (instead of
# defaulting to OpenAI, which would require a separate OPENAI_API_KEY)
ragas_llm = LangchainLLMWrapper(llm)
ragas_embeddings = LangchainEmbeddingsWrapper(embedding_model)

result = evaluate(
    dataset = dataset,
    metrics = [
        context_precision,
        context_recall,
        answer_relevancy,
        faithfulness],
    llm = ragas_llm,
    embeddings = ragas_embeddings,
)


In [ ]:
result

In [ ]:
results = result.to_pandas()
results


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

heatmap_data = results[['context_precision', 'context_recall', 'answer_relevancy', 'faithfulness']]

plt.figure(figsize=(10, 8))
sns.heatmap(heatmap_data, annot=True, fmt=".2f", linewidths=.5)

plt.yticks(ticks=range(len(results['question'])), labels=results['question'], rotation=0)
plt.show()
